In [ ]:
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_distances
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.cluster import KMeans
from pathlib import Path
from IPython.display import display

# Install sentence-transformers in the notebook environment if it's missing.
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

nltk.download("all")


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nl

True

In [ ]:
data_file = "../data/all_genres_dataset.csv"
data = pd.read_csv(data_file)
# Remove songs with missing lyrics and reset the index
data = data.dropna(subset=['lyrics']).reset_index(drop=True)
data

,Unnamed: 0,id,Title,Primary Artists,Featuring Artists,Danceability,Energy,Speechiness,Acousticness,Instrumentalness,...,detected_languages,num_languages,language_confidences,artist_id,explicit,track_popularity,artist_followers,artist_popularity,Loudness,super_genre
0,0,47EiUVwUp4C9fGccaPuUCS,dákiti,"bad bunny, jhayco",NaN,0.721491,0.573158,0.033898,0.403421,0.000000,...,SPANISH,0.020408,SPANISH: 1.0,4q3ewBCX7sLwd24euuV69X,True,0.847826,0.704726,0.95,0.543100,dance
1,1,7igeByaBM0MgGsgXtNxDJ7,positions,ariana grande,NaN,0.726974,0.804238,0.067797,0.470825,0.000000,...,ENGLISH,0.020408,ENGLISH: 1.0,66CXWjxzNUsdJxJ2JdwvnR,True,0.000000,0.739731,0.90,0.784316,pop
2,2,3qQbCzHBycnDpGskqOWY0E,ella baila sola,"eslabon armado, peso pluma",NaN,0.652412,0.759839,0.011653,0.485915,0.000000,...,SPANISH,0.020408,SPANISH: 1.0,0XeEobZplHxzM9QzFQWLiR,False,0.793478,0.044379,0.75,0.765338,folk
3,3,0Yde1VrWbGF2Tus2AQhMuT,circles,post malone,NaN,0.690789,0.757820,0.016949,0.223340,0.003138,...,ENGLISH,0.020408,ENGLISH: 1.0,246dkjvS1zLTtiykXe5h60,False,0.000000,0.328871,0.85,0.845849,dance
4,4,0TK2YIli7K1leLovkQiNik,señorita,"camila cabello, shawn mendes",NaN,0.752193,0.539859,0.007415,0.037223,0.000000,...,ENGLISH,0.020408,ENGLISH: 1.0,7n2wHs1TKAczGzO7Dd2rGr,False,0.858696,0.314936,0.81,0.726060,pop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8029,9154,7wBJfHzpfI3032CSD7CE2m,stargazing,travis scott,NaN,0.453947,0.791120,0.023305,0.009054,0.000000,...,ENGLISH,0.020408,ENGLISH: 1.0,0Y5tJX1MQlPlqiwlOH1tJY,True,0.793478,0.282803,0.89,0.832969,rap
8030,9155,7yotKA30dwTKNEGomV9ZsI,kod,j. cole,NaN,0.822368,0.785066,0.139831,0.211268,0.000000,...,ENGLISH,0.020408,ENGLISH: 1.0,6l3HvQ5sa6mXTsMTB19rO5,True,0.652174,0.186467,0.82,0.715365,rap
8031,9156,5FOCkGt7bp22nBVfQ4WSIT,indecente,anitta,NaN,0.878289,0.602422,0.111229,0.160966,0.001046,...,SPANISH,0.020408,SPANISH: 1.0,7FNnA9vBm6EKceENgCGRMb,False,0.521739,0.091350,0.78,0.767977,pop
8032,9157,7zQirOExB0VR8yWUOqYeio,dolla sign slime (feat. megan thee stallion),lil nas x,megan thee stallion,0.932018,0.532795,0.254237,0.050302,0.000000,...,ENGLISH,0.020408,ENGLISH: 1.0,7jVv8c5Fj3E9VhNjxT4snq,True,0.543478,0.084908,0.71,0.727107,trap


In [7]:
# Build semantic embeddings for lyrics, find similar songs, and train a simple prediction model.
# Uses sentence-transformers if available, otherwise falls back to TF-IDF.
lyrics = data['lyrics'].fillna("").astype(str).tolist()

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
embeddings_sparse = vectorizer.fit_transform(lyrics)
embeddings = embeddings_sparse  # keep sparse matrix
embeddings_is_sparse = True

nn = NearestNeighbors(n_neighbors=6, metric='cosine', n_jobs=-1)
nn.fit(embeddings)

def get_similar(song_idx: int, k: int = 5):
    """
    Return k most similar songs (index, distance, title, primary artists).
    Excludes the query itself from the results.
    """
    query = embeddings[song_idx] if embeddings_is_sparse else embeddings[song_idx]
    distances, indices = nn.kneighbors([query] if not embeddings_is_sparse else query, n_neighbors=k+1)
    distances = distances[0]
    indices = indices[0]
    results = []
    for dist, idx in zip(distances, indices):
        if idx == song_idx:
            continue
        row = data.iloc[idx]
        results.append({'idx': int(idx), 'distance': float(dist), 'title': row['Title'], 'artists': row['Primary Artists']})
        if len(results) >= k:
            break 
    return results

In [8]:
def get_songs_by_artist(artist: str) -> pd.DataFrame:
    a = artist.strip().lower()
    prim = data['Primary Artists'].fillna('').astype(str)
    feat = data['Featuring Artists'].fillna('').astype(str)

    pattern = re.escape(a)
    mask_prim = prim.str.lower().str.contains(pattern, na=False)
    mask_feat = feat.str.lower().str.contains(pattern, na=False)

    return data[mask_prim | mask_feat].copy()

def get_songs_by_genre(genre: str) -> pd.DataFrame:
    g = genre.strip().lower()
    genre = data["super_genre"].fillna('').astype(str)

    pattern = re.escape(g)
    mask_genre = genre.str.lower().str.contains(pattern, na=False)

    return data[mask_genre].copy()

def get_lyrics_by_title(title: str, exact: bool = True, first_only: bool = True):
    t = title.strip().lower()
    titles = data['Title'].fillna('').astype(str).str.lower()
    if exact:
        mask = titles == t
    else:
        mask = titles.str.contains(re.escape(t), na=False)
    matches = data[mask]

    if matches.empty:
        return None if first_only else matches
    if first_only:
        row = matches.iloc[0]
        return row['lyrics']
    
    return matches[['lyrics']].copy()

features = ["Danceability", "Energy", "Loudness", "Speechiness", "Acousticness", "Instrumentalness", "Valence"]

In [ ]:
INCLUDE_GENRE_ONEHOT = True  # Set to False to disable genre one-hot inclusion
super_genre_filled = data['super_genre'].fillna('').astype(str).values.reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
genre_onehot = encoder.fit_transform(super_genre_filled) if INCLUDE_GENRE_ONEHOT else None


In [10]:
# Global variable to cache semantic embeddings
semantic_embeddings_cache = None

def ensure_semantic_embeddings(data_file):
    """
    Ensure semantic embeddings are available, either from cache or by computing them.
    Returns dense embeddings for all lyrics in the dataset.
    """
    global semantic_embeddings_cache
    
    if semantic_embeddings_cache is not None:
        return semantic_embeddings_cache
    
    try:
        # Use the data file name to generate a unique embeddings file name
        import os
        base_name = os.path.splitext(os.path.basename(data_file))[0]
        if INCLUDE_GENRE_ONEHOT:
            embeddings_file = f"../data/semantic_embeddings_genre_{base_name}.npy"
        else:
            embeddings_file = f"../data/semantic_embeddings_{base_name}.npy"
        
        if Path(embeddings_file).exists():
            print(f"Loading pre-computed semantic embeddings from {embeddings_file}...")
            semantic_embeddings_cache = np.load(embeddings_file)
            return semantic_embeddings_cache
        
        # Compute semantic embeddings using sentence transformers
        print("Computing semantic embeddings using sentence-transformers...")
        model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Get lyrics and ensure they're strings
        lyrics_list = data['lyrics'].fillna("").astype(str).tolist()
        
        # Compute embeddings in batches to avoid memory issues
        batch_size = 32
        all_embeddings = []
        
        for i in range(0, len(lyrics_list), batch_size):
            batch = lyrics_list[i:i+batch_size]
            batch_embeddings = model.encode(batch, show_progress_bar=True)
            all_embeddings.append(batch_embeddings)
        
        semantic_embeddings_cache = np.vstack(all_embeddings)
        
        # Save for future use
        try:
            np.save(embeddings_file, semantic_embeddings_cache)
            print(f"Saved semantic embeddings to {embeddings_file}")
        except Exception as e:
            print(f"Could not save embeddings: {e}")
        
        return semantic_embeddings_cache
        
    except Exception as e:
        print(f"Error computing semantic embeddings: {e}")
        # Fallback to random embeddings with appropriate dimensions
        np.random.seed(42)
        semantic_embeddings_cache = np.random.randn(len(data), 384)  # all-MiniLM-L6-v2 has 384 dimensions
        return semantic_embeddings_cache

def get_hybrid_similarity_chunked(liked_idx, tfidf_weight=0.5, semantic_weight=0.5, chunk_size=1000):
    """
    Compute hybrid similarity using chunked processing to manage memory.
    """
    n_songs = len(data)
    hybrid_sim_agg = np.zeros(n_songs)
    
    # Get TF-IDF similarities
    if 'embeddings' in globals() and embeddings is not None:
        liked_tfidf = embeddings[liked_idx]
        tfidf_dists = cosine_distances(liked_tfidf, embeddings)
        tfidf_sims = 1 - tfidf_dists
        tfidf_sim_agg = np.median(tfidf_sims, axis=0)
    else:
        tfidf_sim_agg = np.zeros(n_songs)
    
    # Get semantic similarities
    try:
        semantic_embs = ensure_semantic_embeddings(data_file)
        liked_semantic = semantic_embs[liked_idx]
        semantic_dists = cosine_distances(liked_semantic, semantic_embs)
        semantic_sims = 1 - semantic_dists
        semantic_sim_agg = np.median(semantic_sims, axis=0)
    except Exception as e:
        print(f"Error computing semantic similarities: {e}")
        semantic_sim_agg = np.zeros(n_songs)
    
    # Combine TF-IDF and semantic similarities
    hybrid_sim_agg = tfidf_weight * tfidf_sim_agg + semantic_weight * semantic_sim_agg
    
    return hybrid_sim_agg

def get_hybrid_embeddings_sample(sample_indices, tfidf_weight=0.5, semantic_weight=0.5):
    """
    Create hybrid embeddings for a sample of songs by combining TF-IDF, semantic embeddings, and genre one-hot encoding.
    """
    # Get TF-IDF embeddings for sample
    if 'embeddings' in globals() and embeddings is not None:
        tfidf_sample = embeddings[sample_indices]
        if hasattr(tfidf_sample, 'toarray'):
            tfidf_sample = tfidf_sample.toarray()
    else:
        # Fallback to zeros if TF-IDF not available
        tfidf_sample = np.zeros((len(sample_indices), 1000))  # Assume 1000 TF-IDF features
    
    # Get semantic embeddings for sample
    try:
        semantic_embs = ensure_semantic_embeddings(data_file)
        semantic_sample = semantic_embs[sample_indices]
    except Exception as e:
        print(f"Error getting semantic embeddings: {e}")
        semantic_sample = np.zeros((len(sample_indices), 384))  # Default semantic embedding size
    
    # Get genre one-hot encoding for sample, only if enabled
    if INCLUDE_GENRE_ONEHOT and 'genre_onehot' in globals() and genre_onehot is not None:
        genre_sample = genre_onehot[sample_indices]
    else:
        genre_sample = np.zeros((len(sample_indices), 0))  # No genre columns if disabled
    
    # Normalize both embeddings to unit vectors
    from sklearn.preprocessing import normalize
    tfidf_normalized = normalize(tfidf_sample, norm='l2', axis=1)
    semantic_normalized = normalize(semantic_sample, norm='l2', axis=1)
    
    # Combine embeddings with weights
    # We need to handle dimension mismatch - pad the smaller one with zeros
    tfidf_dim = tfidf_normalized.shape[1]
    semantic_dim = semantic_normalized.shape[1]
    
    if tfidf_dim > semantic_dim:
        # Pad semantic embeddings
        padding = np.zeros((len(sample_indices), tfidf_dim - semantic_dim))
        semantic_padded = np.hstack([semantic_normalized, padding])
        hybrid_embeddings = tfidf_weight * tfidf_normalized + semantic_weight * semantic_padded
    elif semantic_dim > tfidf_dim:
        # Pad TF-IDF embeddings
        padding = np.zeros((len(sample_indices), semantic_dim - tfidf_dim))
        tfidf_padded = np.hstack([tfidf_normalized, padding])
        hybrid_embeddings = tfidf_weight * tfidf_padded + semantic_weight * semantic_normalized
    else:
        # Same dimensions
        hybrid_embeddings = tfidf_weight * tfidf_normalized + semantic_weight * semantic_normalized
    
    # Concatenate genre one-hot encoding to the hybrid embedding if enabled
    if genre_sample.shape[1] > 0:
        hybrid_with_genre = np.hstack([hybrid_embeddings, genre_sample])
    else:
        hybrid_with_genre = hybrid_embeddings
    
    return hybrid_with_genre

In [11]:
def get_ranked_recommendations(song_list: pd.DataFrame, n_recs: int = 10, model_type="semantic", 
                             semantic_weight: float = 0.7, hybrid_tfidf_weight: float = 0.5) -> pd.DataFrame:
    if song_list.empty:
        return pd.DataFrame(columns=['Title', 'Primary Artists', 'Featuring Artists', 'Feature_Similarity', 'Semantic_Similarity', 'Combined_Score'])
    
    # Special case for random: just pick random songs with no duplicates
    if model_type == 'random':
        np.random.seed(42)  # For reproducibility
        
        # Get indices of liked songs to exclude them
        liked_idx = song_list.index.to_numpy()
        
        # Exclude liked songs from potential recommendations
        available_mask = ~data.index.isin(liked_idx)
        
        # Exclude items with the same Title + Primary Artists as any liked song
        liked_pairs = set(
            zip(
                song_list['Title'].fillna('').astype(str).str.lower().str.strip(),
                song_list['Primary Artists'].fillna('').astype(str).str.lower().str.strip()
            )
        )
        title_artist_series = pd.Series(
            list(zip(
                data['Title'].fillna('').astype(str).str.lower().str.strip(),
                data['Primary Artists'].fillna('').astype(str).str.lower().str.strip()
            )),
            index=data.index
        )
        available_mask &= ~title_artist_series.isin(liked_pairs)
        
        available_indices = data.index[available_mask].tolist()
        
        # Randomly select songs while avoiding duplicates
        selected_idxs = []
        seen_titles_artists = set()
        
        # Shuffle available indices for random selection
        shuffled_indices = available_indices.copy()
        np.random.shuffle(shuffled_indices)
        
        for idx in shuffled_indices:
            if len(selected_idxs) >= n_recs:
                break
                
            # Get normalized title and primary artist for duplicate detection
            title = data.loc[idx, 'Title']
            primary_artist = data.loc[idx, 'Primary Artists']
            
            # Normalize for comparison (lowercase, strip whitespace, handle NaN)
            normalized_title = str(title).lower().strip() if pd.notna(title) else ''
            normalized_artist = str(primary_artist).lower().strip() if pd.notna(primary_artist) else ''
            
            title_artist_key = (normalized_title, normalized_artist)
            
            # Skip if we've already seen this title-artist combination
            if title_artist_key in seen_titles_artists:
                continue
                
            selected_idxs.append(idx)
            seen_titles_artists.add(title_artist_key)
        
        # Create recommendations DataFrame with random scores
        if selected_idxs:
            recs = data.loc[selected_idxs, ['Title', 'Primary Artists', 'Featuring Artists'] + features].copy()
            # Add dummy similarity scores for consistency with other models
            recs['Feature Similarity'] = np.random.rand(len(selected_idxs))
            recs['Lyric Similarity'] = np.random.rand(len(selected_idxs))
            recs['Combined Score'] = np.random.rand(len(selected_idxs))
        else:
            # Return empty DataFrame if no recommendations found
            recs = pd.DataFrame(columns=['Title', 'Primary Artists', 'Featuring Artists'] + features + 
                              ['Feature Similarity', 'Lyric Similarity', 'Combined Score'])
        
        recs = recs.reset_index().drop(columns=['index'])
        return recs
    
    # Impute & scale audio features on full dataset
    impute_vals = data[features].median()
    X_full_imputed = data[features].fillna(impute_vals)
    scaler = StandardScaler().fit(X_full_imputed)
    X_scaled = pd.DataFrame(scaler.transform(X_full_imputed), index=X_full_imputed.index, columns=features)
    
    # Concatenate genre one-hot encoding to the features for the 'features' model type, only if enabled
    if model_type == 'features' and INCLUDE_GENRE_ONEHOT and 'genre_onehot' in globals() and genre_onehot is not None:
        X_scaled = np.hstack([X_scaled.values, genre_onehot])
        liked_idx = song_list.index.to_numpy()
        liked_feat = X_scaled[liked_idx]
        feat_dists = cosine_distances(liked_feat, X_scaled)
        feat_sims_matrix = 1 - feat_dists
        feat_sim_agg = np.median(feat_sims_matrix, axis=0)
    else:
        liked_idx = song_list.index.to_numpy()
        liked_feat = X_scaled.loc[liked_idx].values
        feat_dists = cosine_distances(liked_feat, X_scaled.values)
        feat_sims_matrix = 1 - feat_dists
        feat_sim_agg = np.median(feat_sims_matrix, axis=0)
    
    # Word similarities - initialize with zeros as default
    semantic_sim_agg = np.zeros(len(data))
    
    # Hybrid: Use combination of TF-IDF and Semantic embeddings (memory-efficient)
    if model_type == 'hybrid':
        semantic_sim_agg = get_hybrid_similarity_chunked(
            liked_idx, 
            tfidf_weight=hybrid_tfidf_weight, 
            semantic_weight=1.0 - hybrid_tfidf_weight
        )
    
    # Semantic: Use semantic word-embeddings for lyrical similarity
    elif model_type == 'semantic':
        try:
            emb_dense = ensure_semantic_embeddings(data_file)
            liked_emb = emb_dense[liked_idx]
            sem_dists = cosine_distances(liked_emb, emb_dense)
            sem_sims_matrix = 1 - sem_dists
            semantic_sim_agg = np.median(sem_sims_matrix, axis=0)
        except Exception as e:
            print(f"Semantic embeddings unavailable; using zeros. Error: {str(e)}")
            semantic_sim_agg = np.zeros(len(data))
    
    # TF-IDF: Use word frequencies for lyrical similarity
    elif model_type == "tf-idf":
        if 'embeddings' in globals() and embeddings is not None:
            liked_emb = embeddings[liked_idx]
            sem_dists = cosine_distances(liked_emb, embeddings)
            sem_sims_matrix = 1 - sem_dists
            semantic_sim_agg = np.median(sem_sims_matrix, axis=0)
        else:
            semantic_sim_agg = np.zeros(len(data))
    
    # Features: Use only the audio features, no lyrical similarity
    elif model_type == 'features':
        semantic_sim_agg = np.zeros(len(data))
    
    # For hybrid model, semantic_weight is ignored (combination already done in embeddings)
    if model_type == 'hybrid':
        combined = 0.7 * semantic_sim_agg + 0.3 * feat_sim_agg  # Default weighting
    else:
        combined = semantic_weight * semantic_sim_agg + (1.0 - semantic_weight) * feat_sim_agg
    
    scores = pd.DataFrame({
        'Feature Similarity': feat_sim_agg,
        'Lyric Similarity': semantic_sim_agg,
        'Combined Score': combined
    }, index=data.index)
    
    # Exclude liked songs
    mask = ~scores.index.isin(liked_idx)
    
    # Exclude any items with 100% lyric similarity
    if 'Lyric Similarity' in scores.columns:
        mask &= scores['Lyric Similarity'] < 1.0
    
    # Exclude items with the same Title + Primary Artists as any liked song
    liked_pairs = set(
        zip(
            song_list['Title'].fillna('').astype(str).str.lower().str.strip(),
            song_list['Primary Artists'].fillna('').astype(str).str.lower().str.strip()
        )
    )
    title_artist_series = pd.Series(
        list(zip(
            data['Title'].fillna('').astype(str).str.lower().str.strip(),
            data['Primary Artists'].fillna('').astype(str).str.lower().str.strip()
        )),
        index=data.index
    )
    mask &= ~title_artist_series.isin(liked_pairs)
    
    # Sort by combined score and select top recommendations while avoiding duplicates
    filtered_scores = scores[mask].sort_values('Combined Score', ascending=False)
    
    selected_idxs = []
    seen_titles_artists = set()
    
    for idx in filtered_scores.index:
        if len(selected_idxs) >= n_recs:
            break
            
        # Get normalized title and primary artist for duplicate detection
        title = data.loc[idx, 'Title']
        primary_artist = data.loc[idx, 'Primary Artists']
        
        # Normalize for comparison (lowercase, strip whitespace, handle NaN)
        normalized_title = str(title).lower().strip() if pd.notna(title) else ''
        normalized_artist = str(primary_artist).lower().strip() if pd.notna(primary_artist) else ''
        
        title_artist_key = (normalized_title, normalized_artist)
        
        # Skip if we've already seen this title-artist combination
        if title_artist_key in seen_titles_artists:
            continue
            
        selected_idxs.append(idx)
        seen_titles_artists.add(title_artist_key)
    
    top = scores.loc[selected_idxs]
    
    recs = data.loc[top.index, ['Title', 'Primary Artists', 'Featuring Artists', 'super_genre'] + features].copy()
    recs.rename(columns={'super_genre': 'Super Genre'}, inplace=True)
    recs['Feature Similarity'] = top['Feature Similarity'].values
    recs['Lyric Similarity'] = top['Lyric Similarity'].values
    recs['Combined Score'] = top['Combined Score'].values
    
    recs = recs.reset_index().drop(columns=['index'])
    return recs

In [12]:
def evaluate_model_quality(model_type='semantic', n_samples=500, hybrid_tfidf_weight=0.5):
    # Sample random songs for evaluation first (before model-specific processing)
    sample_indices = random.sample(range(len(data)), min(n_samples, len(data)))
    sample_data = data.iloc[sample_indices].copy()
    
    # Get genre one-hot encoding for sample, only if enabled
    if INCLUDE_GENRE_ONEHOT and 'genre_onehot' in globals() and genre_onehot is not None:
        genre_sample = genre_onehot[sample_indices]
    else:
        genre_sample = np.zeros((len(sample_indices), 0))  # No genre columns if disabled
    
    # Get embeddings based on model type
    if model_type == 'semantic':
        emb_dense = ensure_semantic_embeddings(data_file)
        sample_embeddings = emb_dense[sample_indices]
    elif model_type == 'tfidf':
        emb = embeddings  # TF-IDF embeddings (sparse)
        emb_dense = emb.toarray() if hasattr(emb, 'toarray') else emb
        sample_embeddings = emb_dense[sample_indices]
    elif model_type == 'hybrid':
        # Create hybrid embeddings for sample only
        emb_dense = get_hybrid_embeddings_sample(
            sample_indices,
            tfidf_weight=hybrid_tfidf_weight, 
            semantic_weight=1.0 - hybrid_tfidf_weight
        )
        sample_embeddings = emb_dense
    elif model_type == 'features':
        # Use only audio features as embeddings
        feature_data = data[features].fillna(data[features].median())
        scaler = StandardScaler()
        emb_dense = scaler.fit_transform(feature_data)
        sample_embeddings = emb_dense[sample_indices]
    elif model_type == 'random':
        # Use random embeddings as baseline
        np.random.seed(42)  # For reproducibility
        # create random "audio feature" samples matching column stats, then combine with random noise
        rng = np.random.RandomState(42)
        feat_df = data[features].dropna()
        means = feat_df.mean().values
        stds = feat_df.std().replace(0, 1e-6).values
        mins = feat_df.min().values
        maxs = feat_df.max().values
        
        rand_feat = rng.normal(loc=means, scale=stds, size=(len(data), len(features)))
        # clip to realistic ranges observed in the dataset
        for j in range(len(features)):
            rand_feat[:, j] = np.clip(rand_feat[:, j], mins[j], maxs[j])
        
        # standardize so feature columns have comparable scale with embedding noise
        scaler = StandardScaler()
        rand_feat_scaled = scaler.fit_transform(rand_feat)
        
        # remaining dimensions filled with Gaussian noise so total dim = 100
        remaining = max(0, 100 - rand_feat_scaled.shape[1])
        noise = rng.randn(len(data), remaining)
        
        # final random embeddings include randomized features + noise
        emb_dense = np.hstack([rand_feat_scaled, noise])
        emb_dense = np.random.randn(len(data), 100)  # 100-dim random vectors
        sample_embeddings = emb_dense[sample_indices]
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
    
    # Concatenate genre one-hot encoding to the embeddings if enabled
    if genre_sample.shape[1] > 0:
        sample_embeddings = np.hstack([sample_embeddings, genre_sample])
    
    results = {}
    
    # 1. Silhouette Score (How well separated are the clusters?)
    n_clusters = 10
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(sample_embeddings)
    
    silhouette_avg = silhouette_score(sample_embeddings, cluster_labels)
    results['silhouette_score'] = silhouette_avg
    
    # 2. Semantic Coherence Test (do similar artists cluster together?)
    artist_coherence_scores = []
    unique_artists = sample_data['Primary Artists'].value_counts()
    
    for artist, count in unique_artists.head(10).items():  # Top 10 artists
        if count >= 2:  # Artist has at least 2 songs
            artist_songs = sample_data[sample_data['Primary Artists'] == artist]
            # Create a mapping from original indices to sample embedding indices
            sample_idx_to_embedding_idx = {orig_idx: emb_idx for emb_idx, orig_idx in enumerate(sample_indices)}
            artist_embedding_indices = [sample_idx_to_embedding_idx[idx] for idx in artist_songs.index if idx in sample_idx_to_embedding_idx]
            
            if len(artist_embedding_indices) >= 2:
                artist_embs = sample_embeddings[artist_embedding_indices]
                # Calculate average pairwise distance within artist
                distances = pairwise_distances(artist_embs, metric='cosine')
                avg_intra_distance = np.mean(distances[np.triu_indices(len(distances), k=1)])
                artist_coherence_scores.append(1 - avg_intra_distance)  # Convert to similarity
    
    avg_artist_coherence = np.mean(artist_coherence_scores) if artist_coherence_scores else 0
    results['artist_coherence'] = avg_artist_coherence
    
    # 3. Genre Coherence Test (do songs from the same genre cluster together?)
    genre_coherence_scores = []
    if 'super_genre' in sample_data.columns:
        unique_genres = sample_data['super_genre'].value_counts()
        
        for genre, count in unique_genres.head(10).items():  # Top 10 genres
            if count >= 2 and pd.notna(genre):  # Genre has at least 2 songs and is not NaN
                genre_songs = sample_data[sample_data['super_genre'] == genre]
                # Create a mapping from original indices to sample embedding indices
                sample_idx_to_embedding_idx = {orig_idx: emb_idx for emb_idx, orig_idx in enumerate(sample_indices)}
                genre_embedding_indices = [sample_idx_to_embedding_idx[idx] for idx in genre_songs.index if idx in sample_idx_to_embedding_idx]
                
                if len(genre_embedding_indices) >= 2:
                    genre_embs = sample_embeddings[genre_embedding_indices]
                    # Calculate average pairwise distance within genre
                    distances = pairwise_distances(genre_embs, metric='cosine')
                    avg_intra_distance = np.mean(distances[np.triu_indices(len(distances), k=1)])
                    coherence_score = 1 - avg_intra_distance
                    genre_coherence_scores.append(coherence_score)
    
    avg_genre_coherence = np.mean(genre_coherence_scores) if genre_coherence_scores else 0
    results['genre_coherence'] = avg_genre_coherence
    
    # 4. Diversity (how varied are the recommendations?)
    
    # Calculate average pairwise distances
    sample_distances = pairwise_distances(sample_embeddings, metric='cosine')
    avg_distance = np.mean(sample_distances[np.triu_indices(len(sample_distances), k=1)])
    results['average_diversity'] = avg_distance
    
    # Calculate embedding dimensionality efficiency
    explained_variance = np.var(sample_embeddings, axis=0).sum()
    embedding_efficiency = silhouette_avg / (explained_variance + 1e-8)  # Add small epsilon to avoid division by zero
    results['embedding_efficiency'] = embedding_efficiency
    
    return results

In [13]:
evaluation_complete = False

In [ ]:
def calculate_composite_score(results, primary_weight=0.4, secondary_weight=0.3, tertiary_weight=0.3):
    # Clamp values to [0, 1] for each metric
    primary = min(max(results.get('silhouette_score', 0), 0), 1)
    secondary = min(max(results.get('artist_coherence', 0), 0), 1)
    tertiary = min(max(results.get('genre_coherence', 0), 0), 1)
    total_weight = primary_weight + secondary_weight + tertiary_weight
    if total_weight == 0:
        return 0.0
    # Normalize weights to sum to 1
    primary_weight /= total_weight
    secondary_weight /= total_weight
    tertiary_weight /= total_weight
    return primary_weight * primary + secondary_weight * secondary + tertiary_weight * tertiary

# Run evaluation for all models, preventing duplicate execution
if not evaluation_complete:
    n_samples = data.shape[0]
    print(f"Evaluating {n_samples} random songs from the dataset...")

    print(" - Getting TF-IDF Results...")
    tfidf_results = evaluate_model_quality('tfidf', n_samples=n_samples)

    print(" - Getting Semantic Results...")
    semantic_results = evaluate_model_quality('semantic', n_samples=n_samples)

    # Find the best hybrid_tfidf_weight based on composite score
    print(" + Getting Hybrid Results")
    best_hybrid_weight_value = None
    best_hybrid_results = None
    best_score = None
    for idx, w in enumerate([0.1 * n for n in range(0, 11)]):
        print(f" ↳ Trying hybrid_tfidf_weight={w:.1f}")
        hybrid_results = evaluate_model_quality('hybrid', n_samples=n_samples, hybrid_tfidf_weight=w)
        score = calculate_composite_score(hybrid_results)
        if idx == 0 or score > best_score:
            best_score = score
            best_hybrid_weight_value = w
            best_hybrid_results = hybrid_results
    print(f" - Best hybrid_tfidf_weight: {best_hybrid_weight_value:.2f} (composite score={best_score:.4f})")

    print(" - Getting Features-Only Results...")
    features_results = evaluate_model_quality('features', n_samples=n_samples)
    print(" - Getting Random Baseline Results...")
    random_results = evaluate_model_quality('random', n_samples=n_samples)
    
    evaluation_complete = True
else:
    print("Evaluation already complete. Using cached results.")

Evaluating 8034 random songs from the dataset...
 - Getting TF-IDF Results...
 - Getting Semantic Results...
Loading pre-computed semantic embeddings from ../data/semantic_embeddings_genre_all_genres_dataset.npy...
 - Getting Semantic Results...
Loading pre-computed semantic embeddings from ../data/semantic_embeddings_genre_all_genres_dataset.npy...
 + Getting Hybrid Results
 ↳ Trying hybrid_tfidf_weight=0.0
 + Getting Hybrid Results
 ↳ Trying hybrid_tfidf_weight=0.0
 ↳ Trying hybrid_tfidf_weight=0.1
 ↳ Trying hybrid_tfidf_weight=0.1
 ↳ Trying hybrid_tfidf_weight=0.2
 ↳ Trying hybrid_tfidf_weight=0.2
 ↳ Trying hybrid_tfidf_weight=0.3
 ↳ Trying hybrid_tfidf_weight=0.3
 ↳ Trying hybrid_tfidf_weight=0.4
 ↳ Trying hybrid_tfidf_weight=0.4
 ↳ Trying hybrid_tfidf_weight=0.5
 ↳ Trying hybrid_tfidf_weight=0.5
 ↳ Trying hybrid_tfidf_weight=0.6
 ↳ Trying hybrid_tfidf_weight=0.6
 ↳ Trying hybrid_tfidf_weight=0.7
 ↳ Trying hybrid_tfidf_weight=0.7
 ↳ Trying hybrid_tfidf_weight=0.8
 ↳ Trying hybrid_t

In [15]:
# Mapping for display names and formatting rules (reuse existing `metrics` if present)
metrics = [
    ('silhouette_score', 'Silhouette Score', '{:.4f}'),
    ('average_diversity', 'Mean Pairwise Distance', '{:.4f}'),
    ('embedding_efficiency', 'Embedding Efficiency (Sil/Var)', '{:.4f}'),
    ('artist_coherence', 'Artist Coherence Score', '{:.4f}'),
    ('genre_coherence', 'Genre Coherence Score', '{:.4f}')
]

GREEN = '\033[92m'
WHITE = '\033[97m'
GREY = '\033[90m'
BOLD = '\033[1m'
RESET = '\033[0m'

# Display results table
if 'tfidf_results' in globals():
    header = f"{'\nMetric':40s} {'Features':12s} {'TF-IDF':12s} {'Semantic':12s} {'Hybrid':12s} {'Random':12s} {'Best'}"
    print(header)
    print("-" * len(header))

for key, display_name, _ in metrics:
    features_val = features_results.get(key, 0)
    tfidf_val = tfidf_results.get(key, 0)
    semantic_val = semantic_results.get(key, 0)
    hybrid_val = hybrid_results.get(key, 0)
    random_val = random_results.get(key, 0)
    
    # Format all values
    fmt = next(fmt for k, _, fmt in metrics if k == key)
    features_str = fmt.format(features_val)
    tfidf_str = fmt.format(tfidf_val)
    semantic_str = fmt.format(semantic_val)
    hybrid_str = fmt.format(hybrid_val)
    random_str = fmt.format(random_val)
    
    # Find the best value
    values = [features_val, tfidf_val, semantic_val, hybrid_val, random_val]
    names = ['Features', 'TF-IDF', 'Semantic', 'Hybrid', 'Random']
    if key == 'average_diversity':
        best_idx = values.index(min(values))
    else:
        best_idx = values.index(max(values))
    best_name = names[best_idx]
    
    # Color by rank: best=GREEN+BOLD, 2nd=DIM_GREEN, else=GREY
    raw_displays = [features_str, tfidf_str, semantic_str, hybrid_str, random_str]
    # Determine ranking (higher is better except for average_diversity where lower is better)
    reverse = (key != 'average_diversity')
    sorted_indices = sorted(range(len(values)), key=lambda i: values[i], reverse=reverse)
    
    displays = []
    for i, val in enumerate(raw_displays):
        # find rank of this column (0 = best)
        rank = sorted_indices.index(i)
        if rank == 0:
            color = GREEN
            prefix = BOLD
        elif rank == 1:
            color = WHITE
            prefix = ''
        else:
            color = GREY
            prefix = ''
        displays.append(f"{prefix}{color}{val:12s}{RESET}")
    
    print(f"{display_name:40s} {" ".join(displays)} {best_name}")

# Model Types:
#  - TF-IDF: Term Frequency-Inverse Document Frequency on lyrics
#  - Semantic: Sentence-transformer embeddings (all-MiniLM-L6-v2) on lyrics  
#  - Hybrid: Blend of TF-IDF and Semantic scores for Lyrics
#  - Features: Audio features only (Danceability, Energy, etc.)
#  - Random: Random embeddings (baseline control)
#
# Metrics:
#  - Silhouette Score: Measures how well-separated the clusters are in embedding space
#  - Artist Coherence: Measures whether songs by the same artist cluster together
#  - Genre Coherence: Measures whether songs from the same genre cluster together
#  - Mean Pairwise Distance: Measures how spread out/diverse songs are in the embedding space
#  - Embedding Efficiency: silhouette_score/variance, high variance may be noise if clustering is poor (as in random)
#
# Parameter Selection:
#  - Uses composite scoring: Primary (Silhouette Score), Secondary (Artist Coherence), Tertiary (Genre Coherence)
#  - Weights: Primary=1.0, Secondary=0.3, Tertiary=0.1


Metric                                  Features     TF-IDF       Semantic     Hybrid       Random       Best
--------------------------------------------------------------------------------------------------------------
Silhouette Score                         0.1319       0.1833       0.2552       0.2841       0.0065       Hybrid
Mean Pairwise Distance                   0.9591       0.8978       0.7500       0.8744       0.9982       Semantic
Embedding Efficiency (Sil/Var)           0.0168       0.1021       0.1701       0.1083       0.0001       Semantic
Artist Coherence Score                   0.3369       0.4933       0.6717       0.6018       0.0088       Semantic
Genre Coherence Score                    0.3522       0.5240       0.6927       0.6826       0.0100       Semantic


In [16]:
artist = "slipknot"
artist_songs = get_songs_by_artist(artist)
liked_songs = artist_songs.head(1) if len(artist_songs) > 1 else artist_songs

ranked_recs_tfidf = get_ranked_recommendations(liked_songs, n_recs=10, model_type="tf-idf")
ranked_recs_semantic = get_ranked_recommendations(liked_songs, n_recs=10, model_type="semantic")
ranked_recs_hybrid = get_ranked_recommendations(liked_songs, n_recs=10, model_type="hybrid", hybrid_tfidf_weight=best_hybrid_weight_value)
ranked_recs_features = get_ranked_recommendations(liked_songs, n_recs=10, model_type="features")
ranked_recs_random = get_ranked_recommendations(liked_songs, n_recs=10, model_type="random")

print(f"Artist: {artist}")
display(liked_songs["Title"])

print(f"Semantic + Feature Recommendations:")
display(ranked_recs_semantic)

print(f"TF-IDF + Feature Recommendations:")
display(ranked_recs_tfidf)

print(f"Hybrid TF-IDF/Semantic ({best_hybrid_weight_value:.2f} TF-IDF / {1.0 - best_hybrid_weight_value:.2f} Semantic) + Feature Recommendations")
display(ranked_recs_hybrid)

print("Feature Only Recommendations")
display(ranked_recs_features)

print("Random Recommendations")
display(ranked_recs_random)

print()

Artist: slipknot


1221    critical darling
Name: Title, dtype: object

Semantic + Feature Recommendations:


,Title,Primary Artists,Featuring Artists,Super Genre,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,unsainted,slipknot,NaN,metal,0.388158,0.947528,0.869607,0.037076,0.000000,0.016736,0.232218,0.957975,0.623781,0.724039
1,waiting for the end,linkin park,NaN,metal,0.453947,0.964682,0.812534,0.116525,0.042254,0.000000,0.310669,0.928342,0.607187,0.703533
2,notion,the rare occasions,NaN,electronic,0.258772,0.885974,0.826825,0.062500,0.067404,0.001046,0.299163,0.964590,0.553636,0.676922
3,change (taylor's version),taylor swift,NaN,pop,0.467105,0.817356,0.815993,0.012712,0.000000,0.000000,0.332636,0.913407,0.573122,0.675207
4,jesus lord pt 2,kanye west,NaN,rap,0.494518,0.727548,0.699254,0.076271,0.071429,0.019874,0.125523,0.871469,0.580683,0.667919
5,jesus lord,kanye west,NaN,rap,0.442982,0.714430,0.691198,0.110169,0.098592,0.008368,0.091004,0.863941,0.577363,0.663336
6,black hole sun,soundgarden,NaN,rock,0.303728,0.830474,0.756372,0.020127,0.000000,0.000000,0.126569,0.967860,0.523141,0.656556
7,show me how to live,audioslave,NaN,rock,0.394737,0.856710,0.824868,0.027542,0.000000,0.000000,0.357741,0.934600,0.511028,0.638099
8,too late,the weeknd,NaN,pop,0.619518,0.865792,0.763290,0.116525,0.052314,0.000000,0.209205,0.821234,0.557797,0.636828
9,teardrops,bring me the horizon,NaN,rock,0.464912,0.919273,0.849672,0.091102,0.000000,0.000000,0.392259,0.890442,0.526562,0.635726


TF-IDF + Feature Recommendations:


...

,Title,Primary Artists,Featuring Artists,Super Genre,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,unsainted,slipknot,NaN,metal,0.388158,0.947528,0.869607,0.037076,0.000000,0.016736,0.232218,0.957975,0.460555,0.609781
1,waiting for the end,linkin park,NaN,metal,0.453947,0.964682,0.812534,0.116525,0.042254,0.000000,0.310669,0.928342,0.462373,0.602164
2,notion,the rare occasions,NaN,electronic,0.258772,0.885974,0.826825,0.062500,0.067404,0.001046,0.299163,0.964590,0.413897,0.579105
3,change (taylor's version),taylor swift,NaN,pop,0.467105,0.817356,0.815993,0.012712,0.000000,0.000000,0.332636,0.913407,0.430520,0.575386
4,jesus lord pt 2,kanye west,NaN,rap,0.494518,0.727548,0.699254,0.076271,0.071429,0.019874,0.125523,0.871469,0.447035,0.574365
5,jesus lord,kanye west,NaN,rap,0.442982,0.714430,0.691198,0.110169,0.098592,0.008368,0.091004,0.863941,0.442729,0.569092
6,black hole sun,soundgarden,NaN,rock,0.303728,0.830474,0.756372,0.020127,0.000000,0.000000,0.126569,0.967860,0.375115,0.552939
7,show me how to live,audioslave,NaN,rock,0.394737,0.856710,0.824868,0.027542,0.000000,0.000000,0.357741,0.934600,0.373236,0.541645
8,too late,the weeknd,NaN,pop,0.619518,0.865792,0.763290,0.116525,0.052314,0.000000,0.209205,0.821234,0.420461,0.540693
9,teardrops,bring me the horizon,NaN,rock,0.464912,0.919273,0.849672,0.091102,0.000000,0.000000,0.392259,0.890442,0.389746,0.539954


Feature Only Recommendations


,Title,Primary Artists,Featuring Artists,Super Genre,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,protect the land,system of a down,NaN,metal,0.393640,0.889001,0.819270,0.012712,0.000000,0.000000,0.232218,0.965415,0.0,0.289624
1,lux æterna,metallica,NaN,metal,0.378289,0.998991,0.865556,0.061441,0.000000,0.001046,0.226987,0.960811,0.0,0.288243
2,unsainted,slipknot,NaN,metal,0.388158,0.947528,0.869607,0.037076,0.000000,0.016736,0.232218,0.959717,0.0,0.287915
3,weit weg,rammstein,NaN,metal,0.304825,0.835520,0.768387,0.030720,0.011066,0.000000,0.289749,0.955891,0.0,0.286767
4,numb,linkin park,NaN,metal,0.463816,0.865792,0.811897,0.016949,0.005030,0.000000,0.226987,0.945588,0.0,0.283676
5,armee der tristen,rammstein,NaN,metal,0.410088,0.904137,0.755279,0.028602,0.000000,0.000000,0.322176,0.936684,0.0,0.281005
6,waiting for the end,linkin park,NaN,metal,0.453947,0.964682,0.812534,0.116525,0.042254,0.000000,0.310669,0.929126,0.0,0.278738
7,blah blah blah,armin van buuren,NaN,electronic,0.334430,0.966700,0.822911,0.048729,0.009054,0.004184,0.140167,0.922487,0.0,0.276746
8,mr. brightside,the killers,NaN,alternative,0.310307,0.927346,0.830694,0.062500,0.001006,0.000000,0.215481,0.917522,0.0,0.275257
9,down under (feat. colin hay),luude,colin hay,electronic,0.257675,0.862765,0.813808,0.154661,0.012072,0.001046,0.013598,0.912848,0.0,0.273854


Random Recommendations


,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,better,khalid,NaN,0.573465,0.551968,0.533133,0.079449,0.077465,0.349372,0.089958,0.913876,0.102529,0.572761
1,what are you so afraid of,xxxtentacion,NaN,0.620614,0.309788,0.550246,0.008475,0.778672,0.490586,0.335774,0.981045,0.011918,0.137227
2,désolé,"mero, nimo",NaN,0.747807,0.619576,0.690060,0.255297,0.274648,0.000000,0.573222,0.661811,0.139692,0.937661
3,take my breath,the weeknd,NaN,0.685307,0.770938,0.675314,0.015890,0.013078,0.002092,0.343096,0.816890,0.715736,0.351930
4,i wanna be yours,arctic monkeys,NaN,0.428728,0.415742,0.575596,0.004237,0.136821,0.023013,0.473849,0.749546,0.972794,0.446149
5,race my mind,drake,NaN,0.455044,0.635721,0.673676,0.252119,0.180080,0.000000,0.791841,0.699534,0.983065,0.671660
6,diosa,myke towers,NaN,0.798246,0.746720,0.831012,0.043432,0.400402,0.006276,0.229079,0.306931,0.493979,0.333902
7,ppp,kevin roldan,NaN,0.781798,0.650858,0.689650,0.092161,0.365191,0.000000,0.717573,0.057094,0.869352,0.254259
8,tally,blackpink,NaN,0.674342,0.637740,0.780175,0.009534,0.008048,0.000000,0.419456,0.343052,0.736826,0.631049
9,no heart,"21 savage, metro boomin",NaN,0.913377,0.442987,0.605316,0.383475,0.136821,0.000000,0.682008,0.666936,0.501040,0.024984


In [ ]:
# Get unique non-empty genres from the dataset
genres = set(data['super_genre'].dropna().astype(str).str.strip().unique())
occurrences_per_genre = data['super_genre'].fillna('').astype(str).str.strip().value_counts()

# Show recommendations for the top song in each genre
for genre in sorted(genres):
    count = int(occurrences_per_genre.get(genre, 0))
    top_in_genre = get_songs_by_genre(genre).head(1)
    recs = get_ranked_recommendations(top_in_genre, n_recs=20, model_type="semantic")
    if not top_in_genre.empty:
        title = top_in_genre.iloc[0]['Title']
        artists = top_in_genre.iloc[0]['Primary Artists']
        # ensure strings and handle missing values
        title = '' if pd.isna(title) else str(title)
        artists = '' if pd.isna(artists) else str(artists)
        print(f"{title} — {artists} | {genre}")
    display(recs)

mr. brightside — the killers | alternative


,Title,Primary Artists,Featuring Artists,Super Genre,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,more than life (feat. glaive),machine gun kelly,glaive,electronic,0.471491,0.797175,0.757191,0.067797,0.118712,0.000000,0.405858,0.977392,0.566658,0.689878
1,when i'm alone,post malone,NaN,pop,0.451754,0.656912,0.722328,0.032839,0.035211,0.000000,0.083682,0.800297,0.631404,0.682072
2,bloody valentine,machine gun kelly,NaN,rock,0.490132,0.858729,0.808484,0.062500,0.000000,0.000000,0.464435,0.881019,0.587870,0.675814
3,pill breaker (feat. blackbear & machine gun ke...,"travis barker, trippie redd","blackbear, machine gun kelly",rap,0.497807,0.792129,0.715638,0.060381,0.016097,0.000000,0.219665,0.939686,0.552461,0.668628
4,escapism.,"070 shake, raye",NaN,rap,0.509868,0.743693,0.757191,0.097458,0.138833,0.000000,0.234310,0.929992,0.551707,0.665192
5,lovers rock,tv girl,NaN,pop,0.532895,0.873865,0.757965,0.019068,0.000000,0.005230,0.571130,0.758086,0.618917,0.660668
6,me & you together song,the 1975,NaN,rock,0.218202,0.764884,0.734162,0.013771,0.000000,0.002092,0.384937,0.912087,0.551254,0.659504
7,everlong,foo fighters,NaN,rock,0.372807,0.883956,0.748726,0.015890,0.000000,0.000000,0.353556,0.975724,0.521415,0.657707
8,escapism. - sped up,"070 shake, raye",NaN,rap,0.396930,0.769929,0.757328,0.189619,0.086519,0.000000,0.368201,0.881554,0.553867,0.652173
9,t r a n s p a r e n t s o u l feat. travis barker,willow,travis barker,electronic,0.330044,0.832492,0.764109,0.038136,0.002012,0.000000,0.186192,0.987203,0.507923,0.651707


last night — morgan wallen | country


,Title,Primary Artists,Featuring Artists,Super Genre,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,say you won't let go,james arthur,NaN,pop,0.312500,0.557013,0.664209,0.039195,0.699195,0.000000,0.489540,0.870655,0.696376,0.748660
1,whiskey'd my way,morgan wallen,NaN,country,0.405702,0.517659,0.710859,0.005297,0.606640,0.002092,0.310669,0.855568,0.698193,0.745405
2,thought you should know,morgan wallen,NaN,country,0.501096,0.679112,0.720553,0.006356,0.509054,0.000000,0.504184,0.969581,0.628402,0.730755
3,devil don't know,morgan wallen,NaN,country,0.501096,0.664985,0.743856,0.005297,0.379276,0.000000,0.310669,0.887319,0.663271,0.730485
4,stay,post malone,NaN,pop,0.475877,0.479314,0.754324,0.005297,0.451710,0.000000,0.338912,0.781117,0.681177,0.711159
5,gotta be you,one direction,NaN,pop,0.462719,0.583249,0.794693,0.003178,0.376258,0.000000,0.603556,0.869008,0.631428,0.702702
6,thinkin' bout me,morgan wallen,NaN,country,0.639254,0.758829,0.738076,0.009534,0.494970,0.000000,0.421548,0.736055,0.686523,0.701383
7,it's gotta be you,isaiah firebrace,NaN,other,0.382675,0.746720,0.790096,0.016949,0.222334,0.000000,0.587866,0.793115,0.653955,0.695703
8,over you,the kid laroi,NaN,rap,0.417763,0.719475,0.769525,0.090042,0.433602,0.000000,0.260460,0.835115,0.625089,0.688097
9,push my luck,the chainsmokers,NaN,pop,0.560307,0.643794,0.708948,0.019068,0.289738,0.000000,0.527197,0.851945,0.617021,0.687498


dákiti — bad bunny, jhayco | dance


...

,Title,Primary Artists,Featuring Artists,Super Genre,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,ghostin,ariana grande,NaN,pop,0.354167,0.356206,0.623430,0.007415,0.424547,0.000000,0.088912,0.948482,0.610289,0.711747
1,hard feelings/loveless,lorde,NaN,rap,0.453947,0.443996,0.502139,0.096398,0.033199,0.010460,0.161088,0.888629,0.634224,0.710545
2,ghost town,benson boone,NaN,r&b,0.348684,0.332997,0.542873,0.057203,0.391348,0.000000,0.094142,0.981027,0.592803,0.709270
3,paradise,khalid,NaN,rap,0.557018,0.451060,0.565902,0.135593,0.207243,0.003138,0.519874,0.758817,0.682907,0.705680
4,bigger than the whole sky,taylor swift,NaN,pop,0.353070,0.223007,0.431504,0.031780,0.834004,0.003138,0.041841,0.881717,0.627015,0.703426
5,new normal,khalid,NaN,rap,0.347588,0.533804,0.565128,0.051907,0.081489,0.000000,0.228033,0.924600,0.599888,0.697302
6,burn,juice wrld,NaN,rap,0.308114,0.474268,0.576142,0.022246,0.149899,0.035565,0.013598,0.932355,0.594347,0.695750
7,snooze,sza,NaN,r&b,0.532895,0.550959,0.671810,0.116525,0.141851,0.000000,0.382845,0.873782,0.619364,0.695689
8,good days,sza,NaN,r&b,0.425439,0.651867,0.599945,0.037076,0.420523,0.000000,0.373431,0.813651,0.641964,0.693470
9,waiting on a miracle,stephanie beatriz,NaN,other,0.446272,0.553986,0.630211,0.049788,0.172032,0.000000,0.252092,0.942259,0.582086,0.690138


santa claus is coming to town — the jackson 5 | soul


,Title,Primary Artists,Featuring Artists,Super Genre,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,santa claus is comin' to town,frank sinatra,NaN,jazz,0.617325,0.174571,0.446614,0.046610,0.667002,0.000000,0.645397,0.635834,0.795632,0.747693
1,a holly jolly christmas,burl ives,NaN,other,0.668860,0.373360,0.406699,0.008475,0.582495,0.000000,0.901674,0.890511,0.526774,0.635895
2,run rudolph run,chuck berry,NaN,rock,0.666667,0.716448,0.518068,0.073093,0.816901,0.000000,0.973849,0.746914,0.577289,0.628177
3,my only wish (this year),britney spears,NaN,pop,0.650219,0.647830,0.519479,0.005297,0.204225,0.000000,0.713389,0.922055,0.490567,0.620013
4,wonderful christmastime [edited version] - rem...,paul mccartney,NaN,soul,0.768640,0.510595,0.595212,0.010593,0.323944,0.000000,0.779289,0.914040,0.488472,0.616142
5,wonderful christmastime - remastered 2011 / ed...,paul mccartney,NaN,soul,0.768640,0.510595,0.595212,0.010593,0.323944,0.000000,0.779289,0.914040,0.488472,0.616142
6,wonderful christmastime - edited version / rem...,paul mccartney,NaN,rap,0.768640,0.510595,0.595212,0.010593,0.323944,0.000000,0.779289,0.914040,0.488472,0.616142
7,i won't back down,tom petty,NaN,rock,0.758772,0.475277,0.387084,0.011653,0.050302,0.000000,0.982218,0.936266,0.472964,0.611954
8,ain't no mountain high enough,"marvin gaye, tammi terrell",NaN,soul,0.646930,0.600404,0.506190,0.010593,0.432596,0.000000,0.809623,0.945964,0.461899,0.607119
9,here comes santa claus (right down santa claus...,elvis presley,NaN,rock,0.550439,0.233098,0.228017,0.021186,0.973843,0.018828,0.839958,0.739911,0.544146,0.602876


loco contigo — dj snake, j balvin, tyga | trap


,Title,Primary Artists,Featuring Artists,Super Genre,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,mami chula,"jhayco, quevedo",NaN,electronic,0.792763,0.702321,0.837111,0.024364,0.216298,0.000000,0.312762,0.966846,0.739594,0.807769
1,dictadura,anuel aa,NaN,dance,0.821272,0.709384,0.789232,0.064619,0.170020,0.000000,0.449791,0.924935,0.756819,0.807254
2,amargura,karol g,NaN,dance,0.928728,0.697275,0.848170,0.055085,0.184105,0.000000,0.542887,0.869928,0.748969,0.785257
3,todo de ti,rauw alejandro,NaN,dance,0.775219,0.720484,0.836474,0.030720,0.303823,0.000000,0.324268,0.971466,0.695132,0.778032
4,fiel,"jhay cortez, los legendarios, wisin",NaN,electronic,0.850877,0.702321,0.800337,0.040254,0.409457,0.000000,0.501046,0.881882,0.727445,0.773776
5,culpables,"anuel aa, karol g",NaN,dance,0.718202,0.803229,0.867013,0.048729,0.280684,0.000000,0.407950,0.860760,0.729568,0.768926
6,diosa,myke towers,NaN,electronic,0.798246,0.746720,0.831012,0.043432,0.400402,0.006276,0.229079,0.924800,0.698360,0.766292
7,casualidad,mora,NaN,dance,0.801535,0.758829,0.750865,0.019068,0.276660,0.000000,0.435146,0.876246,0.714814,0.763244
8,jangueo,"anuel aa, tego calderon",NaN,dance,0.866228,0.729566,0.866466,0.039195,0.398390,0.000000,0.651674,0.791078,0.748766,0.761460
9,buscate otro,"atomic otro way, jc la nevula",NaN,rap,0.822368,0.688194,0.776807,0.140890,0.247485,0.000000,0.391213,0.857710,0.719138,0.760710
